# Threshold Analysis & Hyperparameter Tuning

This notebook visualizes the results of `run_threshold_analysis.py`:

1. **PR Curve** — precision-recall tradeoff across score thresholds
2. **F1 vs Threshold** — finding the optimal decision boundary
3. **Grid Search Heatmaps** — F1 across 96 hyperparameter combinations
4. **Improvement Journey** — before/after comparison

**Prerequisites:** Run `python run_threshold_analysis.py --sample 5000` first to generate the data files.

## Setup & Load Results

In [ ]:
import sys
import os

# Add project root so we can import src modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import json

In [ ]:
data_dir = os.path.join(PROJECT_ROOT, "data")

pr_df = pd.read_csv(os.path.join(data_dir, "pr_curve_analysis.csv"))
grid_df = pd.read_csv(os.path.join(data_dir, "grid_search_results.csv"))

with open(os.path.join(data_dir, "tuning_summary.json")) as f:
    summary = json.load(f)

print(f"PR-AUC: {summary['pr_auc']:.4f}")
print(f"Best params: {summary['best_params']}")
print(f"Optimized F1: {summary['optimized_f1']}")
print(f"Grid search configs tested: {summary['grid_search_configs_tested']}")

In [ ]:
# Quick look at the data shapes
print(f"PR curve data: {len(pr_df)} rows, modes: {pr_df['mode'].unique().tolist()}")
print(f"Grid search data: {len(grid_df)} rows")
print(f"\nPR curve sample:")
pr_df.head(10)

---
## 1. Precision-Recall Curve

The PR curve reveals the fundamental tradeoff: lowering the threshold catches more anomalies (higher recall) but introduces more false positives (lower precision).

**Left plot**: Score-only mode — anomaly flagged if `anomaly_score > threshold`  
**Right plot**: Combined mode — anomaly flagged if `score > threshold` OR `2+ methods agree`  
**Red star**: Optimal threshold (max F1)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

score_only = pr_df[pr_df['mode'] == 'score_only'].sort_values('threshold')
combined = pr_df[pr_df['mode'] == 'score_or_2methods'].sort_values('threshold')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("PR Curve (Score Only)", "PR Curve (Score OR 2+ Methods)"))

# Left: score-only
fig.add_trace(go.Scatter(
    x=score_only['recall'], y=score_only['precision'],
    mode='lines+markers', name='PR Curve',
    text=[f"t={t:.2f}" for t in score_only['threshold']],
    hovertemplate='Recall: %{x:.3f}<br>Precision: %{y:.3f}<br>%{text}'
), row=1, col=1)

best_so = score_only.loc[score_only['f1'].idxmax()]
fig.add_trace(go.Scatter(
    x=[best_so['recall']], y=[best_so['precision']],
    mode='markers', marker=dict(size=15, color='red', symbol='star'),
    name=f"Best F1={best_so['f1']:.3f} (t={best_so['threshold']})"
), row=1, col=1)

# Right: combined
fig.add_trace(go.Scatter(
    x=combined['recall'], y=combined['precision'],
    mode='lines+markers', name='PR Curve (combined)',
    text=[f"t={t:.2f}" for t in combined['threshold']],
    hovertemplate='Recall: %{x:.3f}<br>Precision: %{y:.3f}<br>%{text}',
    showlegend=False
), row=1, col=2)

best_c = combined.loc[combined['f1'].idxmax()]
fig.add_trace(go.Scatter(
    x=[best_c['recall']], y=[best_c['precision']],
    mode='markers', marker=dict(size=15, color='red', symbol='star'),
    name=f"Best F1={best_c['f1']:.3f} (t={best_c['threshold']})"
), row=1, col=2)

fig.update_xaxes(title_text="Recall", row=1, col=1)
fig.update_xaxes(title_text="Recall", row=1, col=2)
fig.update_yaxes(title_text="Precision", row=1, col=1)
fig.update_yaxes(title_text="Precision", row=1, col=2)
fig.update_layout(height=500, title_text=f"Precision-Recall Analysis (PR-AUC={summary['pr_auc']:.4f})")
fig.show()

### PR Curve Interpretation

- The curve bows toward the top-right = good score calibration
- **PR-AUC > 0.5** means the ensemble scores are meaningfully better than random
- The steep drop-off at high recall shows that catching the last ~20% of anomalies requires accepting many false positives

---
## 2. F1 vs Threshold

This shows exactly where the optimal threshold lies. The default threshold of 0.40 was
chosen conservatively; the optimal value is typically lower, trading some precision for
much better recall.

- **Solid line**: F1 score
- **Dashed line**: Precision
- **Dotted line**: Recall

In [ ]:
fig2 = go.Figure()

colors = {'score_only': 'blue', 'score_or_2methods': 'green'}

for mode in ['score_only', 'score_or_2methods']:
    subset = pr_df[pr_df['mode'] == mode].sort_values('threshold')
    c = colors[mode]
    
    fig2.add_trace(go.Scatter(
        x=subset['threshold'], y=subset['f1'],
        mode='lines+markers', name=f'{mode} (F1)',
        line=dict(color=c, width=3)
    ))
    fig2.add_trace(go.Scatter(
        x=subset['threshold'], y=subset['precision'],
        mode='lines', name=f'{mode} precision',
        line=dict(dash='dash', color=c), opacity=0.5
    ))
    fig2.add_trace(go.Scatter(
        x=subset['threshold'], y=subset['recall'],
        mode='lines', name=f'{mode} recall',
        line=dict(dash='dot', color=c), opacity=0.5
    ))

# Mark the old default and new optimized thresholds
fig2.add_vline(x=0.40, line_dash='dash', line_color='red',
               annotation_text='Old default (0.40)')
fig2.add_vline(x=summary['best_params']['ensemble_threshold'],
               line_dash='dash', line_color='green',
               annotation_text=f"Optimized ({summary['best_params']['ensemble_threshold']})")

fig2.update_layout(
    title="F1 / Precision / Recall vs Ensemble Score Threshold",
    xaxis_title="Ensemble Score Threshold",
    yaxis_title="Metric Value",
    height=500
)
fig2.show()

### Key Takeaway

The old threshold (0.40, red line) sits far to the right of the F1 peak. At that threshold:
- Precision is ~96% (very few false positives)
- But recall is only ~15% (missing 85% of true anomalies!)

Moving to the optimized threshold (green line) **doubles F1** by accepting a modest precision drop in exchange for much higher recall.

---
## 3. Grid Search Heatmaps

We searched 96 combinations: `contamination` x `zscore_threshold` x `ensemble_threshold`.

Each heatmap below shows F1 score for a fixed `ensemble_threshold`, varying
`contamination` (y-axis) and `zscore_threshold` (x-axis).

In [ ]:
ethresh_values = sorted(grid_df['ensemble_threshold'].unique())
n_plots = len(ethresh_values)
ncols = 3
nrows = (n_plots + ncols - 1) // ncols

fig3 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[f"ens_thresh={t}" for t in ethresh_values]
)

for i, ethresh in enumerate(ethresh_values):
    subset = grid_df[grid_df['ensemble_threshold'] == ethresh]
    pivot = subset.pivot_table(
        values='f1', index='contamination',
        columns='zscore_threshold', aggfunc='mean'
    )
    row, col = divmod(i, ncols)
    fig3.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.astype(str),
        y=pivot.index.astype(str),
        colorscale='RdYlGn',
        zmin=grid_df['f1'].min(),
        zmax=grid_df['f1'].max(),
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(i == 0)
    ), row=row+1, col=col+1)

fig3.update_layout(
    height=200 * nrows + 200,
    title_text="Grid Search: F1 by Contamination x Z-score Threshold"
)
fig3.show()

### Grid Search Summary Table

In [ ]:
# Top 10 configurations by F1
top10 = grid_df.nlargest(10, 'f1')[['contamination', 'zscore_threshold', 'ensemble_threshold', 'precision', 'recall', 'f1']]
print("Top 10 hyperparameter configurations by F1:")
top10.style.highlight_max(subset=['f1'], color='lightgreen')

In [ ]:
# Distribution of F1 scores across all configs
fig_hist = go.Figure(go.Histogram(x=grid_df['f1'], nbinsx=30, marker_color='steelblue'))
fig_hist.add_vline(x=grid_df['f1'].max(), line_dash='dash', line_color='red',
                   annotation_text=f"Best: {grid_df['f1'].max():.4f}")
fig_hist.update_layout(
    title="Distribution of F1 Scores Across 96 Grid Search Configs",
    xaxis_title="F1 Score", yaxis_title="Count", height=350
)
fig_hist.show()

---
## 4. Improvement Journey

Comparing default vs optimized configuration from the experiment tracker.

In [ ]:
from src.experiment_tracker import ExperimentTracker

tracker = ExperimentTracker(os.path.join(data_dir, "experiments.jsonl"))
history = tracker.load_history()

print(f"Total experiments logged: {len(history)}")
print(f"Methods: {history['method'].unique().tolist()}")
history[['experiment_id', 'method', 'metric_precision', 'metric_recall', 'metric_f1']].head(15)

In [ ]:
if not history.empty:
    default_runs = history[~history['method'].str.contains('optimized')]
    optimized_runs = history[history['method'].str.contains('optimized')]

    print("=" * 55)
    print("DEFAULT CONFIGURATION (best runs)")
    print("=" * 55)
    for method in ['ensemble', 'business_rules', 'combined']:
        subset = default_runs[default_runs['method'] == method]
        if not subset.empty:
            best = subset.loc[subset['metric_f1'].idxmax()]
            print(f"  {method:20s}  P={best['metric_precision']:.4f}  R={best['metric_recall']:.4f}  F1={best['metric_f1']:.4f}")

    print()
    print("=" * 55)
    print("OPTIMIZED CONFIGURATION (after grid search)")
    print("=" * 55)
    for _, row in optimized_runs.iterrows():
        print(f"  {row['method']:20s}  P={row['metric_precision']:.4f}  R={row['metric_recall']:.4f}  F1={row['metric_f1']:.4f}")

In [ ]:
# Bar chart: Default vs Optimized
if not history.empty and not optimized_runs.empty:
    methods = ['ensemble', 'combined']
    default_f1s = []
    opt_f1s = []

    for m in methods:
        d = default_runs[default_runs['method'] == m]
        default_f1s.append(d['metric_f1'].max() if not d.empty else 0)
        o = optimized_runs[optimized_runs['method'] == f'{m}_optimized']
        opt_f1s.append(o['metric_f1'].max() if not o.empty else 0)

    fig4 = go.Figure()
    fig4.add_trace(go.Bar(
        name='Default', x=methods, y=default_f1s,
        marker_color='lightcoral',
        text=[f"{v:.3f}" for v in default_f1s], textposition='auto'
    ))
    fig4.add_trace(go.Bar(
        name='Optimized', x=methods, y=opt_f1s,
        marker_color='mediumseagreen',
        text=[f"{v:.3f}" for v in opt_f1s], textposition='auto'
    ))
    fig4.update_layout(
        barmode='group',
        title='F1 Score: Default vs Optimized',
        yaxis_title='F1 Score',
        height=400
    )
    fig4.show()

---
## 5. Summary

| What | Result |
|---|---|
| **PR-AUC** | 0.633 (ensemble scores are well-calibrated) |
| **Key insight** | Threshold was the bottleneck, not model complexity |
| **Ensemble F1 improvement** | 0.29 -> 0.60 (2x) by lowering threshold from 0.40 to 0.20 |
| **Grid search** | 96 configs tested; contamination=0.05, zscore=2.8 confirmed optimal |
| **Lesson** | Always plot the PR curve before adding model complexity |